In [26]:
import os
import shutil
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2
import numpy as np
from tqdm import tqdm

In [27]:
# Path ke dataset mentah
ANNOTATIONS_DIR = '../dataset/annotations'
IMAGES_DIR = '../dataset/images'

# Path untuk output hasil konversi
PROCESSED_DIR = '../dataset/processed_yolo'

CLASS_MAPPING = {
    'With Helmet': 0, 
    'Without Helmet': 1  
}

# Buat folder struktur YOLO
for subset in ['train', 'val', 'test']:
    os.makedirs(f'{PROCESSED_DIR}/{subset}/images', exist_ok=True)
    os.makedirs(f'{PROCESSED_DIR}/{subset}/labels', exist_ok=True)

print("Folder processed ready!")

Folder processed ready!


In [28]:
def convert_xml_to_yolo(xml_path, img_width, img_height):
    """Konversi XML PASCAL VOC ke format YOLO txt"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        print(f"  ERROR parse XML: {e}")
        return []

    yolo_lines = []
    for obj in root.findall('object'):
        # .strip() menghilangkan spasi di depan/belakang
        class_name = obj.find('name').text.strip()
        
        if class_name not in CLASS_MAPPING:
            # Tampilkan kelas yang tidak dikenal
            print(f"  WARNING: Kelas '{class_name}' tidak dikenal di {os.path.basename(xml_path)}")
            continue
        
        class_id = CLASS_MAPPING[class_name]
        box = obj.find('bndbox')
        
        try:
            xmin = float(box.find('xmin').text)
            ymin = float(box.find('ymin').text)
            xmax = float(box.find('xmax').text)
            ymax = float(box.find('ymax').text)
        except:
            print(f"  ERROR: Bounding box invalid di {os.path.basename(xml_path)}")
            continue
        
        # Normalisasi
        x_center = ((xmin + xmax) / 2) / img_width
        y_center = ((ymin + ymax) / 2) / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height
        
        # Clamping
        x_center = max(0, min(1, x_center))
        y_center = max(0, min(1, y_center))
        width = max(0, min(1, width))
        height = max(0, min(1, height))
        
        # CEK: Jika width atau height 0, skip
        if width == 0 or height == 0:
            print(f"  WARNING: Bounding box width/height 0 di {os.path.basename(xml_path)}")
            continue
        
        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return yolo_lines

In [29]:
# Dapatkan semua file
image_files = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
xml_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.xml')]

print(f"Total Gambar: {len(image_files)}")
print(f"Total Anotasi: {len(xml_files)}")

# Split 70:20:10
train_imgs, temp_imgs = train_test_split(image_files, test_size=0.3, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=1/3, random_state=42)

print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")

def process_subset(image_list, subset_name):
    for img_name in tqdm(image_list, desc=f"Processing {subset_name}"):
        # Salin gambar
        src_img_path = os.path.join(IMAGES_DIR, img_name)
        dst_img_path = os.path.join(PROCESSED_DIR, subset_name, 'images', img_name)
        shutil.copy2(src_img_path, dst_img_path)
        
        # Cari file XML
        base_name = os.path.splitext(img_name)[0]
        xml_name = base_name + '.xml'
        src_xml_path = os.path.join(ANNOTATIONS_DIR, xml_name)
        
        if not os.path.exists(src_xml_path):
            continue
        
        # Baca ukuran gambar
        img = cv2.imread(src_img_path)
        if img is None:
            continue
        h, w = img.shape[:2]
        
        # Konversi dan simpan label
        yolo_lines = convert_xml_to_yolo(src_xml_path, w, h)
        txt_name = base_name + '.txt'
        dst_txt_path = os.path.join(PROCESSED_DIR, subset_name, 'labels', txt_name)
        with open(dst_txt_path, 'w') as f:
            f.write('\n'.join(yolo_lines))

# Eksekusi
process_subset(train_imgs, 'train')
process_subset(val_imgs, 'val')
process_subset(test_imgs, 'test')

print("Data selesai dikonversi!")

Total Gambar: 764
Total Anotasi: 764
Train: 534, Val: 153, Test: 77


Processing test: 100%|██████████| 77/77 [00:00<00:00, 106.88it/s]

Data selesai dikonversi!


In [30]:
yaml_content = f"""
path: {os.path.abspath(PROCESSED_DIR)}
train: train/images
val: val/images
test: test/images

nc: 2
names: ['With helmet', 'Without helmet']
"""

yaml_path = os.path.join(PROCESSED_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"data.yaml dibuat di: {yaml_path}")

data.yaml dibuat di: ../dataset/processed_yolo\data.yaml


new


In [31]:
import os

label_dir = '../dataset/processed_yolo/train/labels'

# Cek 5 file pertama
zero_files = 0
total_files = 0
for f in os.listdir(label_dir)[:10]:
    path = os.path.join(label_dir, f)
    size = os.path.getsize(path)
    total_files += 1
    if size == 0:
        zero_files += 1
    print(f"{f}: {size} bytes")

print(f"\nTotal sample: {total_files}, Zero byte files: {zero_files}")

# Baca isi salah satu file yang 0 byte
sample_path = os.path.join(label_dir, 'BikesHelmets0.txt')
if os.path.exists(sample_path):
    with open(sample_path, 'r') as f:
        content = f.read()
    print(f"\nIsi BikesHelmets0.txt: '{content}' (panjang: {len(content)})")

BikesHelmets0.txt: 154 bytes
BikesHelmets1.txt: 37 bytes
BikesHelmets100.txt: 76 bytes
BikesHelmets101.txt: 37 bytes
BikesHelmets102.txt: 37 bytes
BikesHelmets105.txt: 37 bytes
BikesHelmets106.txt: 115 bytes
BikesHelmets108.txt: 37 bytes
BikesHelmets109.txt: 37 bytes
BikesHelmets11.txt: 232 bytes

Total sample: 10, Zero byte files: 0

Isi BikesHelmets0.txt: '0 0.138750 0.496255 0.132500 0.205993
0 0.393750 0.411985 0.177500 0.329588
0 0.558750 0.264045 0.112500 0.205993
0 0.847500 0.232210 0.160000 0.337079' (panjang: 151)


In [32]:
def convert_xml_to_yolo(xml_path, img_width, img_height):
    """Konversi XML PASCAL VOC ke format YOLO txt (dengan error handling)"""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except Exception as e:
        print(f"  ERROR parse XML: {e}")
        return []

    yolo_lines = []
    for obj in root.findall('object'):
        # 🔥 PERBAIKAN: .strip() menghilangkan spasi di depan/belakang
        class_name = obj.find('name').text.strip()
        
        if class_name not in CLASS_MAPPING:
            # 🔥 LOGGING: Tampilkan kelas yang tidak dikenal
            print(f"  WARNING: Kelas '{class_name}' tidak dikenal di {os.path.basename(xml_path)}")
            continue
        
        class_id = CLASS_MAPPING[class_name]
        box = obj.find('bndbox')
        
        try:
            xmin = float(box.find('xmin').text)
            ymin = float(box.find('ymin').text)
            xmax = float(box.find('xmax').text)
            ymax = float(box.find('ymax').text)
        except:
            print(f"  ERROR: Bounding box invalid di {os.path.basename(xml_path)}")
            continue
        
        # Normalisasi
        x_center = ((xmin + xmax) / 2) / img_width
        y_center = ((ymin + ymax) / 2) / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height
        
        # Clamping
        x_center = max(0, min(1, x_center))
        y_center = max(0, min(1, y_center))
        width = max(0, min(1, width))
        height = max(0, min(1, height))
        
        # 🔥 CEK: Jika width atau height 0, skip
        if width == 0 or height == 0:
            print(f"  WARNING: Bounding box width/height 0 di {os.path.basename(xml_path)}")
            continue
        
        yolo_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return yolo_lines

In [25]:
import xml.etree.ElementTree as ET

xml_path = '../dataset/annotations/BikesHelmets0.xml'
tree = ET.parse(xml_path)
root = tree.getroot()

print("=== ISI XML ===\n")
print(ET.tostring(root, encoding='unicode', method='xml'))

# Cek semua elemen object
print("\n=== OBJECTS ===")
for obj in root.findall('object'):
    name = obj.find('name')
    if name is not None:
        print(f"Nama kelas (raw): '{name.text}'")
        print(f"Nama kelas (strip): '{name.text.strip()}'")
    else:
        print("Elemen 'name' tidak ditemukan")
    
    # Cek bounding box
    bndbox = obj.find('bndbox')
    if bndbox is not None:
        xmin = bndbox.find('xmin')
        if xmin is not None:
            print(f"  xmin: {xmin.text}")
        else:
            print("  xmin tidak ditemukan")
    else:
        print("  bndbox tidak ditemukan")

=== ISI XML ===

<annotation>
    <folder>images</folder>
    <filename>BikesHelmets0.png</filename>
    <size>
        <width>400</width>
        <height>267</height>
        <depth>3</depth>
    </size>
    <segmented>0</segmented>
    <object>
        <name>With Helmet</name>
        <pose>Unspecified</pose>
        <truncated>0</truncated>
        <occluded>0</occluded>
        <difficult>0</difficult>
        <bndbox>
            <xmin>29</xmin>
            <ymin>105</ymin>
            <xmax>82</xmax>
            <ymax>160</ymax>
        </bndbox>
    </object>
    <object>
        <name>With Helmet</name>
        <pose>Unspecified</pose>
        <truncated>0</truncated>
        <occluded>0</occluded>
        <difficult>0</difficult>
        <bndbox>
            <xmin>122</xmin>
            <ymin>66</ymin>
            <xmax>193</xmax>
            <ymax>154</ymax>
        </bndbox>
    </object>
    <object>
        <name>With Helmet</name>
        <pose>Unspecified</pose>
        